In [73]:
print("Hello")

# Sustainable mobile phone display


Hello


In [1]:
from langchain_community.document_loaders.csv_loader import CSVLoader
import sys
import csv

csv.field_size_limit(sys.maxsize)

131072

In [2]:
loader = CSVLoader(file_path='./test_summarization_loop/patents_summarized_abstract_clean.csv')

documents = loader.load()

In [44]:
#Embedding model load

from langchain.embeddings import HuggingFaceBgeEmbeddings
from langchain.vectorstores import Chroma
from codecarbon import EmissionsTracker
import time

print("Start of code time: ",time.time())

tracker = EmissionsTracker()
tracker.start()

model_name = "BAAI/bge-large-en-v1.5"
# model_kwargs = {'device': 'cuda'}
encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity
model = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    # model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
    # ,
    # query_instruction="为这个句子生成表示以用于检索相关文章："
)


emissions: float = tracker.stop()
print("Emissions for this code execution is: ", emissions)
print("End of code time: ",time.time())

[codecarbon WARNING @ 00:04:00] Invalid gpu_ids format. Expected a string or a list of ints.
[codecarbon INFO @ 00:04:00] [setup] RAM Tracking...
[codecarbon INFO @ 00:04:00] [setup] GPU Tracking...
[codecarbon INFO @ 00:04:00] Tracking Nvidia GPU via pynvml
[codecarbon INFO @ 00:04:00] [setup] CPU Tracking...
[codecarbon WARNING @ 00:04:00] No CPU tracking mode found. Falling back on CPU constant mode.


Start of code time:  1717801440.4403791


[codecarbon WARNING @ 00:04:01] We saw that you have a Intel(R) Xeon(R) Gold 6330 CPU @ 2.00GHz but we don't know it. Please contact us.
[codecarbon INFO @ 00:04:01] CPU Model on constant consumption mode: Intel(R) Xeon(R) Gold 6330 CPU @ 2.00GHz
[codecarbon INFO @ 00:04:01] >>> Tracker's metadata:
[codecarbon INFO @ 00:04:01]   Platform system: Linux-5.4.0-182-generic-x86_64-with-glibc2.31
[codecarbon INFO @ 00:04:01]   Python version: 3.12.1
[codecarbon INFO @ 00:04:01]   CodeCarbon version: 2.4.1
[codecarbon INFO @ 00:04:01]   Available RAM : 503.339 GB
[codecarbon INFO @ 00:04:01]   CPU count: 112
[codecarbon INFO @ 00:04:01]   CPU model: Intel(R) Xeon(R) Gold 6330 CPU @ 2.00GHz
[codecarbon INFO @ 00:04:01]   GPU count: 2
[codecarbon INFO @ 00:04:01]   GPU model: 2 x NVIDIA A40
[codecarbon INFO @ 00:04:07] Energy consumed for RAM : 0.000116 kWh. RAM Power : 188.75217247009277 W
[codecarbon INFO @ 00:04:07] Energy consumed for all GPUs : 0.000096 kWh. Total GPU Power : 155.277552940

Emissions for this code execution is:  6.225456186088755e-05
End of code time:  1717801447.1427157


In [8]:
print(model)

client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': True}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
) model_name='BAAI/bge-large-en-v1.5' cache_folder=None model_kwargs={} encode_kwargs={'normalize_embeddings': True} query_instruction='Represent this question for searching relevant passages: ' embed_instruction=''


In [9]:
import os # Importing os module for operating system functionalities
import shutil # Importing shutil module for high-level file operations

In [42]:
 # Clear out the existing database directory if it exists
if os.path.exists("./db"):
    shutil.rmtree("./db")


In [ ]:
#Chroma DB saving files and embeddings

from langchain_community.vectorstores import Chroma
from codecarbon import EmissionsTracker
import time

print("Start of code time: ",time.time())

tracker = EmissionsTracker()
tracker.start()

chroma_db = Chroma.from_documents(
    documents, model, persist_directory="./db"
)

chroma_db.persist()

emissions: float = tracker.stop()
print("Emissions for this code execution is: ", emissions)
print("End of code time: ",time.time())


In [ ]:
import torch

# del model

torch.cuda.empty_cache()
torch.cuda.mem_get_info()

In [45]:
from langchain.vectorstores import Chroma
vectordb = Chroma(persist_directory="./db", embedding_function= model)


In [46]:
retriever = vectordb.as_retriever(search_type = "similarity_score_threshold",search_kwargs = {"k": 10, "score_threshold":0.50})
# retriever = vectordb.as_retriever(search_type = "mmr",search_kwargs = {"k": 5, "lambda":0.90})
print(retriever.search_type, retriever.search_kwargs)

similarity_score_threshold {'k': 10, 'score_threshold': 0.5}


In [70]:
retirve_result = retriever.get_relevant_documents("Energy Efficient and Sustainable Touch Screen Display")

In [64]:
len(retirve_result)

10

In [71]:
import re

# Assuming the Document class has an attribute 'page_content' that stores the content
class Document:
    def __init__(self, page_content):
        self.page_content = page_content

for item in retirve_result:
    # Regular expression to capture Patent Number and Summary
    pattern = re.compile(r"PatentNumber: (\d+)\n.*SummaryofPatent: Summary:\n(.*)", re.DOTALL)

    match = pattern.search(item.page_content)  # Use dot notation to access the page_content attribute

    if match:
        patent_number = match.group(1).strip()
        summary = match.group(2).strip()
        print("Patent Number:", patent_number)
        print("Summary:", summary)


Patent Number: 11910672
Summary: The invention aims to improve the energy efficiency of touch screens by overlapping touch lines with power supply lines. The non-display area of the substrate includes touch electrodes, touch lines connected to the electrodes, and first and second power supply lines with different voltage levels. The first power supply line overlaps the second power supply line, and at least one of the touch lines overlaps one of the power supply lines. This design can reduce the energy consumption of the touch screen by minimizing the distance between the touch electrodes and the power supply lines. However, the invention does not provide details on the specific energy efficiency improvements or the potential drawbacks of the design.
Patent Number: 11875732
Summary: The research aims to improve the energy efficiency of display panels by developing a screen saver controller that generates screen saver data based on load and life data of panel blocks. The controller adju

In [72]:
from IPython.display import display, HTML

style = """
<style>
    .research-summary {
        border: 1px solid #ddd;
        border-radius: 8px;
        padding: 15px;
        margin: 10px 0;
        background-color: #f9f9f9;
        color: black;
    }
    .research-summary h3 {
        color: #336;
    }
    .research-summary p {
        text-indent: 20px;
    }
</style>
"""

import re

# Assuming the Document class has an attribute 'page_content' that stores the content
class Document:
    def __init__(self, page_content):
        self.page_content = page_content

for item in retirve_result:
    # Regular expression to capture Patent Number and Summary
    pattern = re.compile(r"PatentNumber: (\d+)\n.*SummaryofPatent: Summary:\n(.*)", re.DOTALL)

    match = pattern.search(item.page_content)  # Use dot notation to access the page_content attribute

    if match:
        contents = ""
        patent_number = match.group(1).strip()
        summary = match.group(2).strip()
        # print("Patent Number:", patent_number)
        # print("Summary:", summary)
        contents += f"""
            <div class="research-summary">
                <h3>Patent Number:{patent_number}</h3>
                <p>{summary}</p>
            </div>
            """
        display(HTML(style + contents))


